# Entregable II — Síntesis

Este cuaderno consolida los resultados del Entregable II en el orden metodológico de la guía del curso: análisis individual de variables, comparación entre los tres experimentos del proyecto y evaluación de reducción de dimensión frente al pipeline completo. Lee las salidas de `notebook/experimentos/` y las deja consolidadas en `outputs/entregable_ii/`.

## 1. Entorno y datos

Misma resolución de raíz que los cuadernos anteriores. Si se ejecuta en Colab clona el repositorio y descarga los CSV.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import datetime as dt


In [2]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Repositorio detectado:', root.name)

Repositorio detectado: LogisTech-Modelos-Simulacion-II


In [3]:
import urllib.request
import zipfile

raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))

CSV disponibles en data/raw: 9


In [4]:
import pandas as pd

from src.geography import attach_order_geography
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.variable_analysis import variable_analysis

In [5]:
tables = load_olist_tables(raw)
orders = build_order_level_dataset(tables)
orders = attach_order_geography(orders, tables)
print(f'Pedidos a nivel orden: {len(orders):,}, columnas: {orders.shape[1]}')

Pedidos a nivel orden: 99,441, columnas: 37


In [6]:
output_dir = root / 'outputs' / 'entregable_ii'
output_dir.mkdir(parents=True, exist_ok=True)
print('Salidas del entregable:', output_dir.relative_to(root))

Salidas del entregable: outputs/entregable_ii


## 1.5 Malla de hiperparámetros explorada

La búsqueda con validación cruzada temporal usa la misma malla en los tres experimentos; sólo cambian los predictores y el criterio de selección.

In [7]:
from src.models import model_catalog

def _mallas(tarea, random_state=42):
    filas = []
    for familia, (_, grid) in model_catalog(tarea, random_state).items():
        for parametro, valores in grid.items():
            filas.append({'tarea': tarea, 'familia': familia,
                          'hiperparámetro': parametro, 'valores': valores})
    return filas

mallas = pd.DataFrame(_mallas('regression') + _mallas('classification'))
mallas


,tarea,familia,hiperparámetro,valores
0,regression,Baseline trivial (mediana),strategy,[median]
1,regression,Ridge (parametrico),alpha,"[0.1, 1.0, 10.0]"
2,regression,KNN (no parametrico),n_neighbors,"[7, 15, 31]"
3,regression,KNN (no parametrico),weights,[distance]
4,regression,Random Forest (ensamble),max_depth,"[12, 24, None]"
5,regression,Random Forest (ensamble),min_samples_leaf,"[1, 2, 5]"
6,regression,MLP (red neuronal),hidden_layer_sizes,"[(48,), (64, 32), (64, 32, 16)]"
7,regression,MLP (red neuronal),alpha,"[0.0001, 0.001]"
8,regression,LinearSVR (SVM),C,"[0.1, 1.0, 10.0]"
9,regression,LinearSVR (SVM),epsilon,"[0.1, 1.0]"


## 2. Análisis individual de variables

Para cada predictor calculamos dos vistas que se complementan:

- **Correlación.** Para variables numéricas, Spearman (captura relación monótona sin asumir linealidad). Para categóricas, V de Cramér contra el objetivo (binario en clasificación, discretizado por cuartiles en regresión).
- **Información mutua.** Mide dependencia no lineal y no monótona. Usa `mutual_info_regression` / `mutual_info_classif` de scikit-learn.

Las variables en el cuartil inferior de información mutua se marcan como `drop_candidate`: son las primeras candidatas a retirar si hace falta compactar el espacio de predictores.

In [8]:
X_reg, y_reg, _ = modeling_frame(orders, 'delivery_time_days')
X_cls, y_cls, _ = modeling_frame(orders, 'late_delivery')
print(f'Regresión: {X_reg.shape}, Clasificación: {X_cls.shape}')

Regresión: (96476, 24), Clasificación: (96476, 24)


In [9]:
va_reg = variable_analysis(X_reg, y_reg, task='regression')
va_reg.to_csv(output_dir / 'variable_analysis_regression.csv', index=False)
va_reg

,feature,type,correlation,correlation_kind,abs_correlation,mutual_information,drop_candidate
0,estimated_delivery_days,numeric,0.522157,spearman,0.522157,0.556524,False
1,purchase_hour,numeric,0.000513,spearman,0.000513,0.339708,False
2,mean_customer_seller_distance_km,numeric,0.542751,spearman,0.542751,0.214197,False
3,max_customer_seller_distance_km,numeric,0.540543,spearman,0.540543,0.212459,False
4,purchase_dayofweek,numeric,0.047753,spearman,0.047753,0.189371,False
5,customer_city,categorical,0.369450,cramers_v,0.369450,0.185753,False
6,customer_state,categorical,0.293871,cramers_v,0.293871,0.157104,False
7,total_freight_value,numeric,0.381459,spearman,0.381459,0.153257,False
8,customer_lat,numeric,0.263057,spearman,0.263057,0.151311,False
9,customer_lon,numeric,0.091113,spearman,0.091113,0.144872,False


In [10]:
va_cls = variable_analysis(X_cls, y_cls, task='classification')
va_cls.to_csv(output_dir / 'variable_analysis_classification.csv', index=False)
va_cls

,feature,type,correlation,correlation_kind,abs_correlation,mutual_information,drop_candidate
0,customer_city,categorical,0.261460,cramers_v,0.261460,0.028936,False
1,seller_count,numeric,-0.028407,spearman,0.028407,0.025178,False
2,payment_records,numeric,-0.004934,spearman,0.004934,0.017844,False
3,sellers_with_coordinates,numeric,-0.027222,spearman,0.027222,0.014445,False
4,purchase_month,numeric,-0.033459,spearman,0.033459,0.013501,False
5,customer_lat,numeric,0.045705,spearman,0.045705,0.013083,False
6,customer_lon,numeric,0.073236,spearman,0.073236,0.012933,False
7,item_count,numeric,-0.019413,spearman,0.019413,0.011370,False
8,customer_state,categorical,0.129663,cramers_v,0.129663,0.007548,False
9,payment_value,numeric,0.024404,spearman,0.024404,0.007355,False


In [11]:
drop_reg = va_reg.loc[va_reg['drop_candidate'], 'feature'].tolist()
drop_cls = va_cls.loc[va_cls['drop_candidate'], 'feature'].tolist()
print(f'Candidatas a eliminar (regresión): {drop_reg}')
print(f'Candidatas a eliminar (clasificación): {drop_cls}')

Candidatas a eliminar (regresión): ['mean_product_width_cm', 'sellers_with_coordinates', 'seller_count', 'payment_installments', 'payment_records', 'item_count']
Candidatas a eliminar (clasificación): ['purchase_hour', 'mean_product_height_cm', 'mean_product_weight_g', 'product_category_name', 'payment_installments', 'payment_type']


## 3. Comparación entre los tres experimentos

Cargamos los `summary.json` de los tres experimentos y los ponemos en una sola tabla. El campeón que elegimos para la discusión del informe sale de aquí.

In [12]:
def read_summary(name):
    return json.loads((root / f'outputs/experimentos/{name}/summary.json').read_text())

exp01 = read_summary('01_baseline_pre_compra')
exp02 = read_summary('02_distancia_geografica')
exp03 = read_summary('03_seleccion_robusta')
[exp01, exp02, exp03]

[{'sample_rows': None,
  'cv_splits': 3,
  'bootstrap_iterations': 500,
  'regression': {'champion': 'LinearSVR (SVM)',
   'test_primary': 3.979647162704609,
   'test_ci95': [3.922299705648949, 4.036764568215699]},
  'classification': {'champion': 'MLP (red neuronal)',
   'test_primary': 0.6747671269952074,
   'test_ci95': [0.6603114890517523, 0.6884271227161061]}},
 {'sample_rows': None,
  'cv_splits': 3,
  'bootstrap_iterations': 500,
  'geo_coverage': 0.9872889452036886,
  'regression': {'champion': 'LinearSVR (SVM)',
   'test_primary': 3.9238500219365964,
   'test_ci95': [3.8680654098084446, 3.9797438886485206]},
  'classification': {'champion': 'Random Forest (ensamble)',
   'test_primary': 0.5325684563965212,
   'test_ci95': [0.5149559281104322, 0.5499237273695717]}},
 {'sample_rows': None,
  'cv_splits': 3,
  'bootstrap_iterations': 500,
  'selection_criterion': 'worst',
  'geo_coverage': 0.9872889452036886,
  'regression': {'champion': 'Random Forest (ensamble)',
   'test_prima

In [13]:
comparison = pd.DataFrame(
    [
        {
            'experimento': '01 baseline',
            'regresión campeón': exp01['regression']['champion'],
            'regresión MAE test': exp01['regression']['test_primary'],
            'clasificación campeón': exp01['classification']['champion'],
            'clasificación ROC-AUC test': exp01['classification']['test_primary'],
        },
        {
            'experimento': '02 + distancia geográfica',
            'regresión campeón': exp02['regression']['champion'],
            'regresión MAE test': exp02['regression']['test_primary'],
            'clasificación campeón': exp02['classification']['champion'],
            'clasificación ROC-AUC test': exp02['classification']['test_primary'],
        },
        {
            'experimento': '03 criterio robusto',
            'regresión campeón': exp03['regression']['champion'],
            'regresión MAE test': exp03['regression']['test_primary'],
            'clasificación campeón': exp03['classification']['champion'],
            'clasificación ROC-AUC test': exp03['classification']['test_primary'],
        },
    ]
)
comparison.to_csv(output_dir / 'experiments_comparison.csv', index=False)
comparison

,experimento,regresión campeón,regresión MAE test,clasificación campeón,clasificación ROC-AUC test
0,01 baseline,LinearSVR (SVM),3.979647,MLP (red neuronal),0.674767
1,02 + distancia geográfica,LinearSVR (SVM),3.923850,Random Forest (ensamble),0.532568
2,03 criterio robusto,Random Forest (ensamble),4.144124,Random Forest (ensamble),0.636528


### Tasa de retraso por partición (shift temporal)

La fracción de pedidos con retraso cambia entre las tres particiones cronológicas. Esta diferencia explica por qué Random Forest gana por CV pero Logística generaliza mejor sobre prueba.

In [14]:
from src.splits import split_train_validation_test

X_shift, y_shift, dates_shift = modeling_frame(orders, 'late_delivery')
parts_shift = split_train_validation_test(X_shift, y_shift, dates_shift)

tasas = pd.DataFrame([
    {'partición': 'train', 'n': len(parts_shift['y_train']),
     'tasa_retraso': float(parts_shift['y_train'].mean())},
    {'partición': 'validation', 'n': len(parts_shift['y_validation']),
     'tasa_retraso': float(parts_shift['y_validation'].mean())},
    {'partición': 'test', 'n': len(parts_shift['y_test']),
     'tasa_retraso': float(parts_shift['y_test'].mean())},
])
tasas['tasa_retraso'] = tasas['tasa_retraso'].round(4)
tasas


,partición,n,tasa_retraso
0,train,57885,0.0788
1,validation,19295,0.1162
2,test,19296,0.0529


### Evolución de los campeones entre iteraciones

Síntesis de los tres experimentos con el IC95 del bootstrap sobre prueba.

In [15]:
evolucion = pd.DataFrame([
    {'iteración': 'Baseline (exp 01)',
     'campeón regresión': exp01['regression']['champion'],
     'MAE test': exp01['regression']['test_primary'],
     'IC95 MAE': exp01['regression']['test_ci95'],
     'campeón clasificación': exp01['classification']['champion'],
     'ROC-AUC test': exp01['classification']['test_primary'],
     'IC95 ROC-AUC': exp01['classification']['test_ci95']},
    {'iteración': '+ Haversine (exp 02)',
     'campeón regresión': exp02['regression']['champion'],
     'MAE test': exp02['regression']['test_primary'],
     'IC95 MAE': exp02['regression']['test_ci95'],
     'campeón clasificación': exp02['classification']['champion'],
     'ROC-AUC test': exp02['classification']['test_primary'],
     'IC95 ROC-AUC': exp02['classification']['test_ci95']},
    {'iteración': 'Peor pliegue (exp 03)',
     'campeón regresión': exp03['regression']['champion'],
     'MAE test': exp03['regression']['test_primary'],
     'IC95 MAE': exp03['regression']['test_ci95'],
     'campeón clasificación': exp03['classification']['champion'],
     'ROC-AUC test': exp03['classification']['test_primary'],
     'IC95 ROC-AUC': exp03['classification']['test_ci95']},
])
evolucion


,iteración,campeón regresión,MAE test,IC95 MAE,campeón clasificación,ROC-AUC test,IC95 ROC-AUC
0,Baseline (exp 01),LinearSVR (SVM),3.979647,"[3.9223, 4.036765]",MLP (red neuronal),0.674767,"[0.660311, 0.688427]"
1,+ Haversine (exp 02),LinearSVR (SVM),3.923850,"[3.868065, 3.979744]",Random Forest (ensamble),0.532568,"[0.514956, 0.549924]"
2,Peor pliegue (exp 03),Random Forest (ensamble),4.144124,"[4.087904, 4.197696]",Random Forest (ensamble),0.636528,"[0.620895, 0.652024]"


El campeón que reportamos como "modelo final" del Entregable II sale del experimento 02 enriquecido con distancia (mejor MAE en test, mismo pipeline). La clasificación mantiene la nota explícita de shift temporal: Random Forest gana por CV pero Logística generaliza mejor en test. El experimento 03 añade matices: con el dataset completo el criterio por peor pliegue cambia la familia campeón en regresión (de LinearSVR a RF, con un costo de 0.22 días de MAE) y, en clasificación, mantiene la familia Random Forest pero elige hiperparámetros más estables que suben el test ROC-AUC en +0.10 dentro de la misma familia. Ninguno de los dos criterios cierra la brecha frente a Logística: el shift que importa sigue estando entre desarrollo y prueba, no entre pliegues de la CV.

## 4. Reducción de dimensión frente al pipeline completo

Las tablas `*_reductions.csv` de cada experimento ya tienen PCA al 95 % de varianza y UMAP a 10 dimensiones evaluadas sobre los dos mejores modelos de cada tarea. Aquí consolidamos las del experimento 02 (campeón) para la discusión del informe; las columnas `train_*` y `validation_*` se añadieron sólo al campeón, de modo que 01 y 03 muestran el esquema previo (métricas sobre prueba únicamente).

In [16]:
def load_reduction(name, task):
    df = pd.read_csv(root / f'outputs/experimentos/{name}/{task}_reductions.csv')
    df.insert(0, 'experimento', name)
    df.insert(1, 'tarea', task)
    return df

reductions = pd.concat(
    [
        load_reduction('02_distancia_geografica', 'regression'),
        load_reduction('02_distancia_geografica', 'classification'),
    ],
    ignore_index=True,
)
reductions.to_csv(output_dir / 'reductions_comparison.csv', index=False)
reductions

,experimento,tarea,reduction,model,original_features,reduced_features,reduction_pct,train_MAE,train_RMSE,train_R2,...,validation_recall,validation_F1,validation_ROC_AUC,validation_avg_precision,test_balanced_accuracy,test_precision,test_recall,test_F1,test_ROC_AUC,test_avg_precision
0,02_distancia_geografica,regression,PCA_95pct,LinearSVR (SVM),117,24,79.487179,5.270977,9.277091,0.163166,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02_distancia_geografica,regression,PCA_95pct,MLP (red neuronal),117,24,79.487179,5.309616,8.768890,0.252338,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02_distancia_geografica,regression,UMAP_10d,LinearSVR (SVM),117,10,91.452991,5.634562,9.628723,0.098526,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02_distancia_geografica,regression,UMAP_10d,MLP (red neuronal),117,10,91.452991,5.799975,9.295696,0.159806,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02_distancia_geografica,classification,PCA_95pct,Random Forest (ensamble),117,24,79.487179,NaN,NaN,NaN,...,0.296032,0.296760,0.692762,0.241708,0.499239,0.052306,0.124388,0.073645,0.593613,0.063048
5,02_distancia_geografica,classification,PCA_95pct,Logistica (parametrico),117,24,79.487179,NaN,NaN,NaN,...,0.686580,0.299873,0.713408,0.246214,0.690617,0.093268,0.834476,0.167783,0.723899,0.109080
6,02_distancia_geografica,classification,UMAP_10d,Random Forest (ensamble),117,10,91.452991,NaN,NaN,NaN,...,0.332144,0.258815,0.618974,0.179784,0.491931,0.048156,0.154750,0.073454,0.435010,0.047927
7,02_distancia_geografica,classification,UMAP_10d,Logistica (parametrico),117,10,91.452991,NaN,NaN,NaN,...,0.587160,0.261777,0.635107,0.172320,0.438287,0.035998,0.248776,0.062895,0.390079,0.043124


En regresión, PCA al 95 % reduce de 117 a 24 columnas (−79 %) con una penalidad mínima en el MAE del campeón LinearSVR (3.96 vs 3.92, +0.04 días) y un R² de 0.11. UMAP a 10 dimensiones comprime más fuerte (−91 %) pero empeora el MAE a 4.44 y hunde el R² por debajo de cero para MLP (−0.28), consistente con que el espacio original está dominado por categóricas y la métrica euclidiana sobre embeddings densos no preserva la geometría relevante.

En clasificación PCA conserva casi intacto el ROC-AUC de Logística (0.724 vs 0.727 sin reducir) y mejora levemente el de Random Forest frente al campeón original (0.594 vs 0.533), porque el PCA actúa como regularizador implícito contra el sobreajuste. UMAP a 10 dimensiones destruye la señal en ambos modelos (ROC-AUC 0.44 y 0.39).

## 5. Síntesis

Guardamos un `summary.json` con los campeones por tarea, el experimento del que salen y la lista de candidatas a eliminar por bajo aporte.

In [17]:
logistica_row = pd.read_csv(
    root / 'outputs' / 'experimentos' / '02_distancia_geografica' / 'classification_ranking.csv'
).query("model.str.startswith('Logistica')").iloc[0]
logistica_test_auc = float(logistica_row['test_ROC_AUC'])
rf_test_auc = float(exp02['classification']['test_primary'])

summary = {
    'fecha': dt.date.today().isoformat(),
    'origen_regresion': {
        'experimento': '02_distancia_geografica',
        'campeón': exp02['regression']['champion'],
        'MAE test (días)': exp02['regression']['test_primary'],
        'IC95 MAE': exp02['regression']['test_ci95'],
    },
    'origen_clasificacion_por_cv': {
        'experimento': '02_distancia_geografica',
        'campeón': exp02['classification']['champion'],
        'ROC-AUC test': exp02['classification']['test_primary'],
        'IC95 ROC-AUC': exp02['classification']['test_ci95'],
    },
    'nota_shift_temporal': (
        f'Logística alcanza ROC-AUC {logistica_test_auc:.3f} en test '
        f'(ver experimento 02), muy por encima del campeón elegido por CV '
        f'({rf_test_auc:.3f}). La diferencia es shift entre desarrollo y '
        'prueba y queda explicitada en el informe.'
    ),
    'candidatas_eliminar_regresion': drop_reg,
    'candidatas_eliminar_clasificacion': drop_cls,
}
(output_dir / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary

{'fecha': '2026-11-18',
 'origen_regresion': {'experimento': '02_distancia_geografica',
  'campeón': 'LinearSVR (SVM)',
  'MAE test (días)': 3.9238500219365964,
  'IC95 MAE': [3.8680654098084446, 3.9797438886485206]},
 'origen_clasificacion_por_cv': {'experimento': '02_distancia_geografica',
  'campeón': 'Random Forest (ensamble)',
  'ROC-AUC test': 0.5325684563965212,
  'IC95 ROC-AUC': [0.5149559281104322, 0.5499237273695717]},
 'nota_shift_temporal': 'Logística alcanza ROC-AUC 0.727 en test (ver experimento 02), muy por encima del campeón elegido por CV (0.533). La diferencia es shift entre desarrollo y prueba y queda explicitada en el informe.',
 'candidatas_eliminar_regresion': ['mean_product_width_cm',
  'sellers_with_coordinates',
  'seller_count',
  'payment_installments',
  'payment_records',
  'item_count'],
 'candidatas_eliminar_clasificacion': ['purchase_hour',
  'mean_product_height_cm',
  'mean_product_weight_g',
  'product_category_name',
  'payment_installments',
  'paym